# 🎬 Movie Recommender: classic vs. deep models on MovieLens (ml-latest-small)

This notebook builds and compares a range of recommenders on the MovieLens *latest-small* dataset
(about 100k ratings, 610 users, 9.7k movies), from a popularity baseline to deep GPU models:

| # | Model | Type |
|---|-------|------|
| 3 | Popularity (IMDb weighted rating) | non-personalised baseline |
| 4 | Content-based (TF-IDF genres + tags) | item features |
| 5 | SVD (Surprise) | matrix factorisation, explicit ratings |
| 6 | Item-KNN (KNNBaseline, pearson_baseline) | neighbourhood CF |
| 7 | Hybrid (content candidates, SVD rerank) | blend |
| 8 | NeuMF (GMF + MLP) | deep CF, explicit ratings, GPU |
| 9 | Two-tower retrieval | deep implicit retrieval, in-batch negatives, GPU |
| 10 | SASRec | sequential transformer, GPU (optional) |

Every model is scored on **the same 80/20 split** (random_state=42). Section timings are printed as we go.

## 1. Setup & data loading
Install Surprise (the only missing dependency), seed everything with 42, find the CSVs under `/kaggle/input`,
check that the GPU really works (fall back to CPU with a warning if it doesn't), and load the data.

In [ ]:
!pip install -q scikit-surprise

In [ ]:
import os, re, time, random, difflib, warnings
from contextlib import contextmanager, nullcontext
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import linear_kernel

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 45)
sns.set_theme(style="whitegrid")

SEED = 42
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
TIME_BUDGET_MIN = 45
NOTEBOOK_START = time.time()
SECTION_TIMES = {}


def seed_everything(seed=SEED):
    # Seed python, numpy and torch (CPU + all GPUs) for reproducibility.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@contextmanager
def timed(name):
    # Context manager that prints and records how long a section took.
    t0 = time.time()
    print(f"▶ {name} ...")
    try:
        yield
    finally:
        dt = time.time() - t0
        SECTION_TIMES[name] = SECTION_TIMES.get(name, 0.0) + dt
        print(f"⏱ {name}: {dt:.1f}s  (elapsed total {(time.time() - NOTEBOOK_START) / 60:.1f} min)")


def elapsed_min():
    # Minutes since the notebook started.
    return (time.time() - NOTEBOOK_START) / 60


seed_everything()
print("torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
def setup_device():
    # Return 'cuda' if a working GPU is present, otherwise 'cpu' (with a warning).
    # Beyond torch.cuda.is_available(), run a real matmul (fp32 + fp16 autocast) because a
    # PyTorch build may report CUDA but lack kernels for the assigned GPU architecture.
    if not torch.cuda.is_available():
        print("WARNING: CUDA not available, running on CPU.")
        return "cpu"
    try:
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        x = torch.randn(512, 512, device="cuda")
        y = (x @ x.T).sum().item()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            z = (x @ x).float().mean().item()
        torch.cuda.synchronize()
        print(f"GPU OK: {name} (compute capability {cap[0]}.{cap[1]}), "
              f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | sanity sums {y:.1f}, {z:.4f}")
        return "cuda"
    except Exception as e:  # e.g. "no kernel image is available for execution on the device"
        print(f"WARNING: CUDA sanity check failed ({type(e).__name__}: {e}). Falling back to CPU.")
        return "cpu"


with timed("1a. GPU check"):
    device = setup_device()
USE_AMP = device == "cuda"


def amp_ctx():
    # Mixed-precision autocast on GPU, no-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


def make_scaler():
    # GradScaler for fp16 training (disabled on CPU); handles old and new torch APIs.
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

In [ ]:
def find_csvs(root="/kaggle/input", names=("movies.csv", "ratings.csv", "tags.csv")):
    # Walk the input directory and return {filename: full_path} for the requested CSVs.
    found = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f in names and f not in found:
                found[f] = os.path.join(dirpath, f)
    missing = set(names) - set(found)
    if missing:
        raise FileNotFoundError(f"Could not find {missing} under {root}")
    return found


with timed("1b. Load data"):
    paths = find_csvs()
    for k, v in paths.items():
        print(f"{k:12s} -> {v}")
    movies = pd.read_csv(paths["movies.csv"])
    ratings = pd.read_csv(paths["ratings.csv"])
    tags = pd.read_csv(paths["tags.csv"])

for name, df in [("movies", movies), ("ratings", ratings), ("tags", tags)]:
    print(f"\n{name}: shape={df.shape}")
    display(df.head())

In [ ]:
# Global index spaces shared by every model: users from ratings, items = full movie catalogue.
movies = movies.reset_index(drop=True)
movies["genre_list"] = movies["genres"].apply(lambda g: [] if g == "(no genres listed)" else g.split("|"))
movie_ids = movies["movieId"].values
mid2idx = {m: i for i, m in enumerate(movie_ids)}
user_ids = np.sort(ratings["userId"].unique())
uid2idx = {u: i for i, u in enumerate(user_ids)}
N_USERS, N_ITEMS = len(user_ids), len(movie_ids)
ratings = ratings[ratings["movieId"].isin(mid2idx)].copy()
ratings["u"] = ratings["userId"].map(uid2idx).astype(np.int64)
ratings["i"] = ratings["movieId"].map(mid2idx).astype(np.int64)
titles = movies["title"].values
print(f"{N_USERS} users, {N_ITEMS} movies in catalogue, {len(ratings)} ratings")

## 2. Exploratory data analysis
Rating distribution, activity per user, the long tail of ratings per movie, genre popularity,
ratings over time, and how sparse the user-item matrix is.

In [ ]:
with timed("2. EDA"):
    n_ratings = len(ratings)
    sparsity = 1 - n_ratings / (N_USERS * N_ITEMS)
    per_user = ratings.groupby("userId").size()
    per_movie = ratings.groupby("movieId").size().sort_values(ascending=False)
    genre_counts = movies["genre_list"].explode().value_counts()
    years = pd.to_datetime(ratings["timestamp"], unit="s").dt.year.value_counts().sort_index()

    print(f"Sparsity: {sparsity * 100:.2f}% of the {N_USERS}x{N_ITEMS} matrix is empty")
    print(f"Ratings/user: min {per_user.min()}, median {per_user.median():.0f}, max {per_user.max()}")
    print(f"Ratings/movie: median {per_movie.median():.0f}; {(per_movie == 1).mean() * 100:.1f}% of rated movies have 1 rating;"
          f" top 10% of movies hold {per_movie.head(len(per_movie) // 10).sum() / n_ratings * 100:.1f}% of ratings")

    fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))
    sns.countplot(x="rating", data=ratings, ax=ax[0, 0], color="#4C72B0")
    ax[0, 0].set_title("Rating distribution")
    ax[0, 1].hist(per_user.values, bins=60, color="#55A868")
    ax[0, 1].set_yscale("log"); ax[0, 1].set_title("Ratings per user"); ax[0, 1].set_xlabel("# ratings")
    ax[0, 2].plot(np.arange(1, len(per_movie) + 1), per_movie.values, color="#C44E52")
    ax[0, 2].set_xscale("log"); ax[0, 2].set_yscale("log")
    ax[0, 2].set_title("Ratings per movie (long tail)"); ax[0, 2].set_xlabel("movie rank"); ax[0, 2].set_ylabel("# ratings")
    genre_counts.head(15).iloc[::-1].plot.barh(ax=ax[1, 0], color="#8172B2")
    ax[1, 0].set_title("Top genres (# movies)")
    years.plot(ax=ax[1, 1], marker="o", color="#CCB974")
    ax[1, 1].set_title("Ratings per year")
    ax[1, 2].axis("off")
    ax[1, 2].text(0.05, 0.5, f"Users: {N_USERS}\nMovies: {N_ITEMS}\nRatings: {n_ratings:,}\n"
                  f"Tags: {len(tags):,}\nSparsity: {sparsity * 100:.2f}%\nMean rating: {ratings.rating.mean():.2f}",
                  fontsize=15, va="center", family="monospace")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/eda.png", dpi=80)
    plt.show()

### Train/test split (shared by every model)
One 80/20 random split with `random_state=42`. Every model trains on `train_df` and is evaluated on `test_df`.
For ranking metrics a test item is **relevant** if its rating is ≥ 4, and items the user already rated in train are excluded
from their recommendations. The deep models carve a further 10% validation split out of `train_df` for early stopping.

In [ ]:
with timed("Split"):
    test_df = ratings.sample(frac=0.2, random_state=SEED)
    train_df = ratings.drop(test_df.index)
    val_df = train_df.sample(frac=0.1, random_state=SEED)      # used only by the deep models
    trainsub_df = train_df.drop(val_df.index)
    GLOBAL_MEAN = train_df["rating"].mean()

    def items_by_user(df):
        # Map user index -> np.array of item indices in df.
        return {u: g["i"].values for u, g in df.groupby("u")}

    train_items = items_by_user(train_df)
    relevant_test = {u: set(g["i"].values) for u, g in test_df[test_df.rating >= 4].groupby("u")}
    EVAL_USERS = np.array(sorted(u for u in relevant_test if u in train_items))
    train_counts_item = np.bincount(train_df["i"].values, minlength=N_ITEMS)
    print(f"train {len(train_df)}, test {len(test_df)} (val {len(val_df)} carved from train); "
          f"{len(EVAL_USERS)} users with >=1 relevant test item")

In [ ]:
def rmse_mae(y_true, y_pred):
    # Return (RMSE, MAE) for two arrays of ratings.
    y_true, y_pred = np.asarray(y_true, float), np.clip(np.asarray(y_pred, float), 0.5, 5.0)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))), float(np.mean(np.abs(y_true - y_pred)))


def topk_from_scores(scores, seen, k=10):
    # Indices of the k best-scoring items, excluding already-seen items.
    s = np.asarray(scores, dtype=np.float64).copy()
    if seen is not None and len(seen):
        s[seen] = -np.inf
    k = min(k, len(s))
    top = np.argpartition(-s, k - 1)[:k]
    top = top[np.argsort(-s[top])]
    return top[np.isfinite(s[top])]


def ranking_metrics(score_fn, users=None, k=10, batch=128):
    # Precision@k, Recall@k, NDCG@k and catalogue coverage on the shared test set.
    # score_fn(user_idx_array) -> (len(users), N_ITEMS) score matrix. Train items are masked.
    users = EVAL_USERS if users is None else users
    P, R, N, rec_items = [], [], [], set()
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    for b in range(0, len(users), batch):
        ub = users[b:b + batch]
        S = np.asarray(score_fn(ub), dtype=np.float64)
        for row, u in enumerate(ub):
            top = topk_from_scores(S[row], train_items.get(u), k)
            rel = relevant_test[u]
            hits = np.array([i in rel for i in top], dtype=float)
            hits = np.pad(hits, (0, k - len(hits)))
            P.append(hits.sum() / k)
            R.append(hits.sum() / len(rel))
            N.append((hits * discounts).sum() / discounts[:min(len(rel), k)].sum())
            rec_items.update(top.tolist())
    return {"Precision@10": np.mean(P), "Recall@10": np.mean(R), "NDCG@10": np.mean(N),
            "Coverage": len(rec_items) / N_ITEMS}


RESULTS = {}


def record(name, metrics):
    # Store and print one model's metrics row.
    RESULTS[name] = {**RESULTS.get(name, {}), **metrics}
    print(name, {k: round(float(v), 4) for k, v in metrics.items() if v is not None and not pd.isna(v)})

## 3. Popularity baseline (IMDb weighted rating)
$WR = \frac{v}{v+m}R + \frac{m}{v+m}C$, where $v$ is the movie's number of ratings, $R$ its mean rating,
$C$ the global mean, and $m$ a minimum-votes threshold (90th percentile of vote counts). Movies with few votes are
pulled toward the global mean. The same list goes to every user, and the WR value doubles as a (damped) rating prediction for RMSE.

In [ ]:
with timed("3. Popularity"):
    stats = train_df.groupby("i")["rating"].agg(["count", "mean"])
    m_votes = float(stats["count"].quantile(0.90))
    C = GLOBAL_MEAN
    wr = np.full(N_ITEMS, C)
    v, R = stats["count"].values, stats["mean"].values
    wr[stats.index.values] = v / (v + m_votes) * R + m_votes / (v + m_votes) * C
    print(f"min votes m = {m_votes:.0f}, global mean C = {C:.3f}")

    top10_pop = np.argsort(-wr)[:10]
    display(pd.DataFrame({"title": titles[top10_pop], "votes": train_counts_item[top10_pop],
                          "weighted_rating": wr[top10_pop].round(3)}))

    record("Popularity", {"RMSE": rmse_mae(test_df.rating, wr[test_df.i])[0],
                          "MAE": rmse_mae(test_df.rating, wr[test_df.i])[1]})
    record("Popularity", ranking_metrics(lambda ub: np.tile(wr, (len(ub), 1))))

## 4. Content-based filtering (TF-IDF on genres + user tags)
Each movie becomes a document made of its genres and the user tags attached to it, vectorised with TF-IDF.
Cosine similarity between these vectors drives `similar_movies()`. Similarities are computed on demand
(one row at a time), so no 9.7k × 9.7k matrix is held in memory. Titles are matched loosely, so `"toy story"`,
`"the matrix"` or a small typo still work. For personalised recommendations, a user profile is the
rating-weighted sum of the TF-IDF vectors of the movies they rated above their own average.

In [ ]:
def norm_title(t):
    # Normalise a title for matching: lowercase, drop year/alt titles, 'Matrix, The' -> 'the matrix'.
    t = str(t).lower()
    t = re.sub(r"\(\s*\d{4}\s*(-\s*\d{0,4})?\s*\)\s*$", "", t)   # trailing year
    t = re.sub(r"\(.*?\)", " ", t).strip()                         # alternative titles
    m = re.match(r"^(.*),\s*(the|a|an|les|la|le|il|el|das|der|die|l')$", t)
    if m:
        t = f"{m.group(2)} {m.group(1)}"
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


NORM_TITLES = np.array([norm_title(t) for t in titles])
POPULARITY = np.bincount(ratings["i"].values, minlength=N_ITEMS)


def find_movie(query, verbose=True):
    # Fuzzy title lookup -> item index (or None). Prefers exact, then substring, then difflib matches;
    # ties are broken by number of ratings so 'godfather' finds The Godfather, not a sequel.
    q = norm_title(query)
    if not q:
        return None
    for cands in (np.where(NORM_TITLES == q)[0],
                  np.array([i for i, t in enumerate(NORM_TITLES) if re.search(rf"\b{re.escape(q)}\b", t)], dtype=int)):
        if len(cands):
            return int(cands[np.argmax(POPULARITY[cands])])
    close = difflib.get_close_matches(q, list(set(NORM_TITLES)), n=3, cutoff=0.6)
    if close:
        cands = np.where(np.isin(NORM_TITLES, close))[0]
        idx = int(cands[np.argmax(POPULARITY[cands])])
        if verbose:
            print(f"(fuzzy match: '{query}' -> '{titles[idx]}')")
        return idx
    return None


def build_item_docs():
    # One text document per movie: genre tokens + cleaned user tags.
    genre_doc = movies["genre_list"].apply(lambda gs: " ".join("genre_" + re.sub(r"[^a-z]", "", g.lower()) for g in gs))
    t = tags.dropna(subset=["tag"]).copy()
    t["tag"] = t["tag"].astype(str).str.lower().str.replace(r"[^a-z0-9 ]", " ", regex=True).str.strip()
    tag_doc = t.groupby("movieId")["tag"].apply(lambda s: " ".join(s))
    tag_doc = movies["movieId"].map(tag_doc).fillna("")
    return genre_doc, tag_doc


with timed("4a. TF-IDF"):
    genre_doc, tag_doc = build_item_docs()
    tfidf = TfidfVectorizer(token_pattern=r"[a-z0-9_]+", min_df=2, sublinear_tf=True)
    X_content = tfidf.fit_transform((genre_doc + " " + genre_doc + " " + tag_doc).values)  # genres x2 weight
    X_content = X_content.tocsr().astype(np.float32)
    print(f"TF-IDF matrix: {X_content.shape}, nnz={X_content.nnz} (sparse, {X_content.data.nbytes / 1e6:.1f} MB)")

In [ ]:
def similar_movies(title, n=10):
    # Top-n movies most similar to `title` by TF-IDF cosine; popularity breaks ties among equal scores.
    idx = find_movie(title)
    if idx is None:
        print(f"Movie '{title}' not found. Try another spelling.")
        return pd.DataFrame(columns=["title", "genres", "similarity"])
    sims = linear_kernel(X_content[idx], X_content).ravel()
    sims[idx] = -1
    order = np.lexsort((-POPULARITY, -np.round(sims, 4)))[:n]
    print(f"Movies similar to: {titles[idx]}")
    return pd.DataFrame({"title": titles[order], "genres": movies["genres"].values[order],
                         "similarity": sims[order].round(3)})


def content_user_profiles(users):
    # Sparse (len(users) x n_features) profiles: sum of item vectors weighted by (rating - user mean), positive part.
    sub = train_df[train_df.u.isin(users)]
    w = sub["rating"] - sub.groupby("u")["rating"].transform("mean")
    w = np.where(w > 0, w, 0.0) + 1e-3 * (sub["rating"].values >= 4)   # keep at least liked items
    row_of = {u: r for r, u in enumerate(users)}
    W = sp.csr_matrix((w, (sub["u"].map(row_of).values, sub["i"].values)), shape=(len(users), N_ITEMS))
    return W @ X_content


def content_scores(users):
    # Content-based score matrix (len(users) x N_ITEMS).
    P = content_user_profiles(users)
    return np.asarray((P @ X_content.T).todense(), dtype=np.float32)


with timed("4b. Content eval"):
    display(similar_movies("toy story"))
    record("Content", {"RMSE": np.nan, "MAE": np.nan})
    record("Content", ranking_metrics(content_scores))

## 5. Collaborative filtering: SVD (Surprise)
Biased matrix factorisation: $\hat r_{ui} = \mu + b_u + b_i + q_i^\top p_u$. A small 3-fold `GridSearchCV` over factors,
learning rate and regularisation runs on the training set only. The best model is refit on all of `train_df`.
For fast full-catalogue ranking, predictions are computed in a vectorised way from the learned factors.

In [ ]:
from surprise import Dataset, Reader, SVD, KNNBaseline, accuracy
from surprise.model_selection import GridSearchCV, KFold

reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[["userId", "movieId", "rating"]], reader)
trainset = train_data.build_full_trainset()
testset = list(zip(test_df.userId.values, test_df.movieId.values, test_df.rating.values))

# global item index -> Surprise inner id (or -1), global user index -> inner id
item_inner = np.array([trainset._raw2inner_id_items.get(m, -1) for m in movie_ids])
user_inner = np.array([trainset._raw2inner_id_users.get(u, -1) for u in user_ids])
print(f"Surprise trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")

In [ ]:
with timed("5a. SVD grid search"):
    param_grid = {"n_factors": [50, 100], "n_epochs": [25], "lr_all": [0.005, 0.01],
                  "reg_all": [0.05, 0.1], "random_state": [SEED]}
    gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=KFold(n_splits=3, random_state=SEED, shuffle=True),
                      n_jobs=-1, joblib_verbose=0)
    gs.fit(train_data)
    best_params = gs.best_params["rmse"]
    print(f"best CV RMSE {gs.best_score['rmse']:.4f} with {best_params}")

with timed("5b. SVD fit + eval"):
    svd = SVD(**best_params)
    svd.fit(trainset)
    preds = svd.test(testset)
    record("SVD", {"RMSE": accuracy.rmse(preds, verbose=False), "MAE": accuracy.mae(preds, verbose=False)})


def svd_scores(users):
    # Vectorised SVD predictions for all catalogue items: mu + b_u + b_i + q_i.p_u (unknown -> 0 terms).
    known = item_inner >= 0
    bi = np.zeros(N_ITEMS); bi[known] = svd.bi[item_inner[known]]
    Q = np.zeros((N_ITEMS, svd.qi.shape[1])); Q[known] = svd.qi[item_inner[known]]
    out = np.empty((len(users), N_ITEMS), dtype=np.float32)
    for r, u in enumerate(users):
        iu = user_inner[u]
        out[r] = trainset.global_mean + bi + (svd.bu[iu] + Q @ svd.pu[iu] if iu >= 0 else 0.0)
    return out


with timed("5c. SVD ranking eval"):
    record("SVD", ranking_metrics(svd_scores))

In [ ]:
def recommend_for_user(user_id, n=10, scorer=None, label="SVD"):
    # Top-n unseen movies for a raw userId using `scorer` (default SVD). Unknown users get the popularity list;
    # users with very few ratings get a warning because their personalised scores are unreliable.
    scorer = scorer or svd_scores
    if user_id not in uid2idx or uid2idx[user_id] not in train_items:
        print(f"User {user_id} unknown (cold start): returning the popularity top-{n}.")
        top = np.argsort(-wr)[:n]
        return pd.DataFrame({"title": titles[top], "score": wr[top].round(3)})
    u = uid2idx[user_id]
    seen = train_items[u]
    if len(seen) < 5:
        print(f"Note: user {user_id} has only {len(seen)} training ratings; recommendations are low-confidence.")
    s = scorer(np.array([u]))[0]
    top = topk_from_scores(s, seen, n)
    return pd.DataFrame({"title": titles[top], "genres": movies["genres"].values[top], f"{label}_score": s[top].round(3)})


display(recommend_for_user(1))
display(recommend_for_user(999999))

## 6. Item-based KNN (KNNBaseline, pearson_baseline)
$\hat r_{ui} = b_{ui} + \frac{\sum_{j \in N^k_i(u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in N^k_i(u)} s_{ij}}$, using the shrunk
Pearson-baseline similarity between items. Surprise stores an item × item similarity matrix (~9k², about 0.6 GB, within budget).
Ranking predictions are vectorised over the user's rated items (top-k positive neighbours), mirroring Surprise's `estimate()`.

In [ ]:
K_NEIGHBORS = 40
with timed("6a. KNNBaseline fit + RMSE"):
    knn = KNNBaseline(k=K_NEIGHBORS, sim_options={"name": "pearson_baseline", "user_based": False},
                      bsl_options={"method": "als", "n_epochs": 10}, verbose=False)
    knn.fit(trainset)
    preds_knn = knn.test(testset)
    record("Item-KNN", {"RMSE": accuracy.rmse(preds_knn, verbose=False), "MAE": accuracy.mae(preds_knn, verbose=False)})
    print(f"similarity matrix {knn.sim.shape}, {knn.sim.nbytes / 1e9:.2f} GB")


def knn_scores(users, k=K_NEIGHBORS):
    # Vectorised KNNBaseline predictions over all catalogue items for the given users.
    mu = trainset.global_mean
    known = item_inner >= 0
    inner_known = item_inner[known]
    bi_all = np.zeros(N_ITEMS); bi_all[known] = knn.bi[inner_known]
    out = np.empty((len(users), N_ITEMS), dtype=np.float32)
    for r, u in enumerate(users):
        iu = user_inner[u]
        if iu < 0:
            out[r] = mu + bi_all
            continue
        rated = np.array(trainset.ur[iu])
        j, rj = rated[:, 0].astype(int), rated[:, 1]
        dev = rj - (mu + knn.bu[iu] + knn.bi[j])
        S = knn.sim[np.ix_(inner_known, j)]                       # (n_known_items, n_rated)
        if S.shape[1] > k:
            idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
            S, D = np.take_along_axis(S, idx, axis=1), dev[idx]
        else:
            D = np.broadcast_to(dev, S.shape)
        Wp = np.where(S > 0, S, 0.0)
        ssum = Wp.sum(1)
        adj = np.divide((Wp * D).sum(1), ssum, out=np.zeros_like(ssum), where=ssum > 0)
        out[r] = mu + knn.bu[iu] + bi_all
        out[r, known] += adj
    return out


with timed("6b. KNN ranking eval"):
    record("Item-KNN", ranking_metrics(knn_scores, batch=32))

## 7. Hybrid: content candidates, SVD rerank
1. **Candidates**: the user's 200 best content-based matches among unseen movies.
2. **Rerank**: `score = α · norm(SVD prediction) + (1 − α) · norm(content similarity)` with min-max normalisation inside the candidate set.

α = 0.7 by default. A small α sweep on the test set is printed for insight only; it is *not* used to pick the reported number.
Explanations come from the liked training movie that is most content-similar to each recommendation.

In [ ]:
HYBRID_CANDIDATES, HYBRID_ALPHA = 200, 0.7


def _minmax(x):
    # Min-max normalise to [0, 1] (constant arrays -> 0.5).
    lo, hi = x.min(), x.max()
    return np.full_like(x, 0.5) if hi - lo < 1e-12 else (x - lo) / (hi - lo)


def hybrid_scores(users, alpha=HYBRID_ALPHA, n_cand=HYBRID_CANDIDATES):
    # Hybrid score matrix: only the top content candidates get finite scores.
    Sc, Ss = content_scores(users), svd_scores(users)
    out = np.full((len(users), N_ITEMS), -np.inf, dtype=np.float32)
    for r, u in enumerate(users):
        cand = topk_from_scores(Sc[r], train_items.get(u), n_cand)
        if len(cand) == 0:
            continue
        out[r, cand] = alpha * _minmax(Ss[r, cand]) + (1 - alpha) * _minmax(Sc[r, cand])
    return out


def explain_hybrid(user_id, n=10, alpha=HYBRID_ALPHA):
    # Hybrid top-n for a raw userId with a 'Because you liked X' explanation for each movie.
    if user_id not in uid2idx or uid2idx[user_id] not in train_items:
        print(f"User {user_id} unknown, no personalised explanation possible.")
        return recommend_for_user(user_id, n)
    u = uid2idx[user_id]
    s = hybrid_scores(np.array([u]), alpha)[0]
    top = topk_from_scores(s, train_items[u], n)
    ud = train_df[train_df.u == u]
    liked = ud[ud.rating >= max(4.0, ud.rating.mean())]["i"].values
    liked = liked if len(liked) else ud.sort_values("rating", ascending=False)["i"].values[:5]
    sims = linear_kernel(X_content[top], X_content[liked])
    because = [f"Because you liked {titles[liked[np.argmax(row)]]} (sim {row.max():.2f})" for row in sims]
    return pd.DataFrame({"title": titles[top], "hybrid_score": s[top].round(3), "why": because})


with timed("7. Hybrid eval"):
    record("Hybrid", {"RMSE": np.nan, "MAE": np.nan})
    record("Hybrid", ranking_metrics(hybrid_scores))
    sweep = {a: ranking_metrics(lambda ub, a=a: hybrid_scores(ub, alpha=a))["NDCG@10"] for a in [0.0, 0.3, 0.5, 0.9, 1.0]}
    print("α sensitivity (NDCG@10, test set, for insight only):", {a: round(v, 4) for a, v in sweep.items()})

## 8. Neural Collaborative Filtering: NeuMF (GPU)
NeuMF fuses a **GMF** branch (element-wise product of user and item embeddings) with an **MLP** branch (concatenated embeddings
through 128→64→32 layers), plus user/item biases and the global mean. It is trained on the **same 80/20 split** for explicit
rating prediction with MSE loss. Settings: embedding dim 64, dropout, AdamW, fp16 autocast, batch 2048, and early stopping on
validation RMSE (a 10% split carved from train). The best checkpoint is restored.

In [ ]:
class NeuMF(nn.Module):
    # GMF + MLP rating predictor with biases.

    def __init__(self, n_users, n_items, dim=64, mlp=(128, 64, 32), dropout=0.3, mean=3.5):
        super().__init__()
        self.ug, self.ig = nn.Embedding(n_users, dim), nn.Embedding(n_items, dim)
        self.um, self.im = nn.Embedding(n_users, dim), nn.Embedding(n_items, dim)
        self.ub, self.ib = nn.Embedding(n_users, 1), nn.Embedding(n_items, 1)
        layers, d = [], 2 * dim
        for h in mlp:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.mlp = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(dim + d, 1)
        self.mean = mean
        for e in (self.ug, self.ig, self.um, self.im):
            nn.init.normal_(e.weight, std=0.01)
        for e in (self.ub, self.ib):
            nn.init.zeros_(e.weight)

    def forward(self, u, i):
        gmf = self.drop(self.ug(u) * self.ig(i))
        mlp = self.mlp(torch.cat([self.um(u), self.im(i)], dim=-1))
        x = self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)
        return x + self.ub(u).squeeze(-1) + self.ib(i).squeeze(-1) + self.mean


def to_tensors(df):
    # (users, items, ratings) tensors on the active device.
    return (torch.as_tensor(df.u.values, device=device), torch.as_tensor(df.i.values, device=device),
            torch.as_tensor(df.rating.values, dtype=torch.float32, device=device))


@torch.no_grad()
def neumf_predict(model, u, i, bs=65536):
    # Batched rating predictions (fp32 output) for index tensors u, i.
    model.eval()
    out = []
    for b in range(0, len(u), bs):
        with amp_ctx():
            out.append(model(u[b:b + bs], i[b:b + bs]).float())
    return torch.cat(out)


def train_neumf(max_epochs=60, patience=5, bs=2048, lr=2e-3, wd=1e-4):
    # Train NeuMF with AdamW + AMP and early stopping on validation RMSE. Returns (model, history).
    seed_everything()
    model = NeuMF(N_USERS, N_ITEMS, mean=float(trainsub_df.rating.mean())).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scaler = make_scaler()
    tu, ti, tr = to_tensors(trainsub_df)
    vu, vi, vr = to_tensors(val_df)
    hist, best, best_state, bad = {"train_rmse": [], "val_rmse": []}, np.inf, None, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(tu), device=device)
        tot = 0.0
        for b in range(0, len(perm), bs):
            idx = perm[b:b + bs]
            with amp_ctx():
                loss = F.mse_loss(model(tu[idx], ti[idx]).float(), tr[idx])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item() * len(idx)
        val_rmse = torch.sqrt(F.mse_loss(neumf_predict(model, vu, vi).clamp(0.5, 5), vr)).item()
        hist["train_rmse"].append(np.sqrt(tot / len(tu)))
        hist["val_rmse"].append(val_rmse)
        if val_rmse < best - 1e-4:
            best, bad = val_rmse, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep % 5 == 0 or bad >= patience:
            print(f"epoch {ep:2d} train RMSE {hist['train_rmse'][-1]:.4f} val RMSE {val_rmse:.4f}")
        if bad >= patience:
            print(f"early stop at epoch {ep}, best val RMSE {best:.4f}")
            break
    model.load_state_dict(best_state)
    return model, hist


with timed("8a. NeuMF train"):
    neumf, neumf_hist = train_neumf()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(neumf_hist["train_rmse"], label="train RMSE"); ax.plot(neumf_hist["val_rmse"], label="val RMSE")
ax.set_xlabel("epoch"); ax.set_ylabel("RMSE"); ax.set_title("NeuMF learning curve"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/neumf_loss.png", dpi=80); plt.show()

In [ ]:
@torch.no_grad()
def neumf_scores(users):
    # NeuMF predicted ratings for all catalogue items (len(users) x N_ITEMS).
    items = torch.arange(N_ITEMS, device=device)
    rows = []
    for u in users:
        rows.append(neumf_predict(neumf, torch.full((N_ITEMS,), int(u), device=device), items))
    return torch.stack(rows).cpu().numpy()


with timed("8b. NeuMF eval"):
    teu, tei, ter = to_tensors(test_df)
    p = neumf_predict(neumf, teu, tei).cpu().numpy()
    rm, ma = rmse_mae(test_df.rating.values, p)
    record("NeuMF", {"RMSE": rm, "MAE": ma})
    record("NeuMF", ranking_metrics(neumf_scores))

## 9. Two-tower retrieval model (GPU, implicit feedback)
Ratings ≥ 4 count as positives. The **user tower** is an ID embedding followed by an MLP. The **item tower** combines an ID embedding,
multi-hot genres and 32-d tag features (TruncatedSVD of tag TF-IDF), passed through an MLP. Both outputs are L2-normalised.
Training uses **in-batch negatives** (sampled softmax with temperature τ chosen on validation from a small grid, plus **logQ correction** for the popularity bias of in-batch sampling): each user's positive item is the target, and the other
items in the batch act as negatives. Recommendations are a dot product against every item embedding, excluding seen movies.
Because the item tower sees content features, never-rated movies still get meaningful embeddings (cold start).
Early stopping uses validation Recall@10.

In [ ]:
with timed("9a. Item features"):
    all_genres = sorted({g for gs in movies["genre_list"] for g in gs})
    G = np.zeros((N_ITEMS, len(all_genres)), dtype=np.float32)
    for r, gs in enumerate(movies["genre_list"]):
        for g in gs:
            G[r, all_genres.index(g)] = 1.0
    tag_tfidf = TfidfVectorizer(token_pattern=r"[a-z0-9]+", min_df=2, sublinear_tf=True).fit_transform(tag_doc.values)
    T = TruncatedSVD(n_components=32, random_state=SEED).fit_transform(tag_tfidf).astype(np.float32)
    T = (T - T.mean(0)) / (T.std(0) + 1e-6)
    T[tag_doc.values == ""] = 0.0
    ITEM_FEATS = torch.as_tensor(np.hstack([G, T]), device=device)
    print(f"item side features: {ITEM_FEATS.shape} ({len(all_genres)} genres + 32 tag dims)")


class TwoTower(nn.Module):
    # User tower (ID embedding + MLP) and item tower (ID embedding + genre/tag features + MLP).

    def __init__(self, n_users, n_items, feats, dim=64, hidden=128, dropout=0.2):
        super().__init__()
        self.feats = feats
        self.uemb, self.iemb = nn.Embedding(n_users, dim), nn.Embedding(n_items, dim)
        nn.init.normal_(self.uemb.weight, std=0.05); nn.init.normal_(self.iemb.weight, std=0.05)
        self.emb_drop = nn.Dropout(dropout)          # discourages pure ID memorisation
        self.utower = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, dim))
        self.itower = nn.Sequential(nn.Linear(dim + feats.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout),
                                    nn.Linear(hidden, dim))

    def user(self, u):
        return F.normalize(self.utower(self.emb_drop(self.uemb(u))).float(), dim=-1)

    def item(self, i):
        return F.normalize(self.itower(torch.cat([self.emb_drop(self.iemb(i)), self.feats[i]], dim=-1)).float(), dim=-1)


@torch.no_grad()
def tt_scores_for(model, users):
    # Dot-product scores of the given users against every item embedding.
    model.eval()
    with amp_ctx():
        ue = model.user(torch.as_tensor(np.asarray(users), device=device))
        ie = model.item(torch.arange(N_ITEMS, device=device))
    return (ue.float() @ ie.float().T).cpu().numpy()


def recall_at_k(model, seen_by_user, rel_by_user, k=10):
    # Mean Recall@k of a two-tower model for users in rel_by_user.
    users = np.array(sorted(u for u in rel_by_user if u in seen_by_user))
    S = tt_scores_for(model, users)
    rec = [len(set(topk_from_scores(S[r], seen_by_user[u], k)) & rel_by_user[u]) / len(rel_by_user[u])
           for r, u in enumerate(users)]
    return float(np.mean(rec))


def train_two_tower(max_epochs=100, patience=8, bs=1024, lr=1e-3, wd=1e-4, tau=0.1, verbose=True):
    # Train with in-batch sampled softmax + logQ correction; early stopping on validation Recall@10.
    # logQ: popular items show up as in-batch negatives more often, which biases the model against them;
    # subtracting log(item frequency) from the training logits removes that sampling bias.
    seed_everything()
    pos = trainsub_df[trainsub_df.rating >= 4]
    pu = torch.as_tensor(pos.u.values, device=device)
    pi = torch.as_tensor(pos.i.values, device=device)
    log_q = torch.log(torch.bincount(pi, minlength=N_ITEMS).float() / len(pi) + 1e-12)
    seen_sub = items_by_user(trainsub_df)
    val_rel = {u: set(g.i.values) for u, g in val_df[val_df.rating >= 4].groupby("u")}
    model = TwoTower(N_USERS, N_ITEMS, ITEM_FEATS).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scaler = make_scaler()
    hist, best, best_state, bad = {"train_loss": [], "val_recall": []}, -1, None, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(pu), device=device)
        tot, nb = 0.0, 0
        for b in range(0, len(perm) - bs // 4, bs):        # drop a tiny last batch
            idx = perm[b:b + bs]
            with amp_ctx():
                ue, ie = model.user(pu[idx]), model.item(pi[idx])
                logits = (ue @ ie.T).float() / tau - log_q[pi[idx]][None, :]
                # mask in-batch 'negatives' that are actually the same item (duplicates)
                same = pi[idx][None, :] == pi[idx][:, None]
                logits = logits.masked_fill(same & ~torch.eye(len(idx), dtype=torch.bool, device=device), -1e4)
                loss = F.cross_entropy(logits.float(), torch.arange(len(idx), device=device))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item(); nb += 1
        vr = recall_at_k(model, seen_sub, val_rel)
        hist["train_loss"].append(tot / max(nb, 1)); hist["val_recall"].append(vr)
        if vr > best + 1e-4:
            best, bad = vr, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose and (ep % 5 == 0 or bad >= patience):
            print(f"epoch {ep:2d} loss {hist['train_loss'][-1]:.4f} val Recall@10 {vr:.4f}")
        if bad >= patience:
            break
    print(f"tau={tau}: stopped at epoch {ep}, best val Recall@10 {best:.4f}")
    model.load_state_dict(best_state)
    hist["best_val_recall"] = best
    return model, hist


with timed("9b. Two-tower train"):
    # small temperature grid, selected on the validation split (never on test)
    runs = {tau: train_two_tower(tau=tau, verbose=False) for tau in [0.05, 0.1, 0.2]}
    best_tau = max(runs, key=lambda t: runs[t][1]["best_val_recall"])
    two_tower, tt_hist = runs[best_tau]
    print(f"selected tau={best_tau}")

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(tt_hist["train_loss"], color="#4C72B0", label="train loss (in-batch softmax)")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
ax2 = ax1.twinx(); ax2.plot(tt_hist["val_recall"], color="#C44E52", label="val Recall@10"); ax2.set_ylabel("Recall@10")
ax1.set_title("Two-tower training"); fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.88))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/two_tower_loss.png", dpi=80); plt.show()

with timed("9c. Two-tower eval"):
    record("Two-tower", {"RMSE": np.nan, "MAE": np.nan})
    record("Two-tower", ranking_metrics(lambda ub: tt_scores_for(two_tower, ub)))

## 10. (Optional) SASRec: sequential transformer
A small causal transformer (2 blocks, 2 heads, d=64) reads each user's **time-ordered** history (last 50 items) and predicts the next
item with a full softmax over the catalogue. Training sequences are sliding windows over each history (stride 10), so batches hold
1024 windows. It runs only if the time budget allows.

* **Leave-last-out** (the standard SASRec protocol): train on all but the last 2 items, validate on the 2nd-to-last (early stopping),
  test on the last one (HR@10, NDCG@10 over the full catalogue, excluding seen). Popularity is shown for reference.
* **Shared test set**: a second model is trained on the time-ordered *train-split* items for the best epoch count, and the hidden state
  after the user's train history scores the catalogue for the common metrics table.

In [ ]:
MAX_LEN = 50


LOSS_LAST = 10        # = window stride: each target is trained on ~once per epoch


class Block(nn.Module):
    # Pre-norm transformer block using scaled_dot_product_attention with an explicit boolean mask.

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.p = heads, dropout
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv, self.proj = nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)
        self.ff = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * dim, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, allowed):
        B, L, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, L, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.p if self.training else 0.0)
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, L, D)))
        return x + self.drop(self.ff(self.ln2(x)))


class SASRec(nn.Module):
    # Self-attentive sequential recommender (causal transformer, shared item embeddings for output).

    def __init__(self, n_items, dim=64, heads=2, layers=2, max_len=MAX_LEN, dropout=0.2):
        super().__init__()
        self.item = nn.Embedding(n_items + 1, dim, padding_idx=0)     # 0 = padding, item k -> k+1
        self.pos = nn.Embedding(max_len, dim)
        nn.init.normal_(self.item.weight, std=0.02); nn.init.normal_(self.pos.weight, std=0.02)
        with torch.no_grad():
            self.item.weight[0].zero_()
        self.blocks = nn.ModuleList([Block(dim, heads, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("tril", torch.tril(torch.ones(max_len, max_len, dtype=torch.bool)))

    def forward(self, seq):
        L = seq.shape[1]
        x = self.drop(self.item(seq) * (self.item.embedding_dim ** 0.5) + self.pos(torch.arange(L, device=seq.device)))
        eye = torch.eye(L, dtype=torch.bool, device=seq.device)
        # causal & non-pad keys; every query may attend to itself so padded rows never become all-masked (NaN)
        allowed = ((self.tril[:L, :L][None] & (seq != 0)[:, None, :]) | eye[None])[:, None]   # (B,1,L,L)
        for blk in self.blocks:
            x = blk(x, allowed)
        return self.norm(x)

    def logits(self, h):
        return h @ self.item.weight[1:].T                                 # (.., n_items)


def left_pad(s, L=MAX_LEN):
    # Last L items of s (shifted +1), left-padded with zeros.
    s = np.asarray(s[-L:]) + 1
    return np.concatenate([np.zeros(L - len(s), dtype=np.int64), s])


def make_windows(seqs, L=MAX_LEN, stride=10):
    # Sliding windows (input, target) over each item sequence for next-item training.
    X, Y = [], []
    for s in seqs:
        s = np.asarray(s)
        ends = list(range(len(s), 1, -stride))
        for e in ends:
            w = s[max(0, e - L - 1):e]
            X.append(left_pad(w[:-1], L)); Y.append(left_pad(w[1:], L))
    return torch.as_tensor(np.array(X), device=device), torch.as_tensor(np.array(Y), device=device)


@torch.no_grad()
def sasrec_scores(model, histories):
    # Next-item scores over the catalogue given each user's history (list of item-index arrays).
    model.eval()
    seq = torch.as_tensor(np.array([left_pad(h) for h in histories]), device=device)
    with amp_ctx():
        h = model(seq)[:, -1]
        return model.logits(h).float().cpu().numpy()


def hr_ndcg_last(score_mat, histories, targets, k=10):
    # HR@k and NDCG@k for a single held-out target per user (seen items masked).
    hr, nd = [], []
    for r, (h, t) in enumerate(zip(histories, targets)):
        top = list(topk_from_scores(score_mat[r], np.asarray(h), k))
        hit = t in top
        hr.append(float(hit)); nd.append(1 / np.log2(top.index(t) + 2) if hit else 0.0)
    return float(np.mean(hr)), float(np.mean(nd))


def train_sasrec(train_seqs, epochs=200, patience=15, val=None, bs=1024, lr=2e-3):
    # Train SASRec on train_seqs; if val=(histories, targets) is given, early-stop on val NDCG@10.
    seed_everything()
    X, Y = make_windows(train_seqs)
    model = SASRec(N_ITEMS).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = make_scaler()
    best, best_state, best_ep, bad = -1, None, epochs, 0
    for ep in range(epochs):
        if elapsed_min() > TIME_BUDGET_MIN - 10:          # protect the overall runtime budget
            print(f"time budget reached, stopping SASRec at epoch {ep}")
            best_ep = min(best_ep, max(ep, 1))
            break
        model.train()
        perm = torch.randperm(len(X), device=device)
        for b in range(0, len(perm), bs):
            idx = perm[b:b + bs]
            with amp_ctx():
                h = model(X[idx])
                y = Y[idx]
                mask = y > 0
                mask[:, :-LOSS_LAST] = False
                loss = F.cross_entropy(model.logits(h[mask]).float(), y[mask] - 1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        if val is not None:
            _, nd = hr_ndcg_last(sasrec_scores(model, val[0]), val[0], val[1])
            if nd > best + 1e-4:
                best, bad, best_ep = nd, 0, ep + 1
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
            if ep % 10 == 0 or bad >= patience:
                print(f"epoch {ep:3d} loss {loss.item():.4f} val NDCG@10 {nd:.4f}")
            if bad >= patience:
                print(f"early stop at epoch {ep}, best epoch {best_ep}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep


RUN_SASREC = elapsed_min() < TIME_BUDGET_MIN - 15
print(f"elapsed {elapsed_min():.1f} min -> {'running' if RUN_SASREC else 'SKIPPING'} SASRec")
if RUN_SASREC:
    with timed("10a. SASRec leave-last-out"):
        full_seqs = {u: g.sort_values("timestamp")["i"].values for u, g in ratings.groupby("u")}
        U = sorted(full_seqs)
        tr_seqs = [full_seqs[u][:-2] for u in U]
        val = ([full_seqs[u][:-2] for u in U], [full_seqs[u][-2] for u in U])
        tst = ([full_seqs[u][:-1] for u in U], [full_seqs[u][-1] for u in U])
        sasrec_llo, best_ep = train_sasrec(tr_seqs, val=val)
        hr, nd = hr_ndcg_last(sasrec_scores(sasrec_llo, tst[0]), tst[0], tst[1])
        pop_counts = np.bincount(np.concatenate(tst[0]), minlength=N_ITEMS).astype(float)
        hr_p, nd_p = hr_ndcg_last(np.tile(pop_counts, (len(U), 1)), tst[0], tst[1])
        print(f"Leave-last-out  SASRec HR@10 {hr:.4f} NDCG@10 {nd:.4f} | Popularity HR@10 {hr_p:.4f} NDCG@10 {nd_p:.4f}")
    with timed("10b. SASRec shared test"):
        train_seq = {u: g.sort_values("timestamp")["i"].values for u, g in train_df.groupby("u")}
        sasrec, _ = train_sasrec([train_seq[u] for u in sorted(train_seq)], epochs=best_ep)
        record("SASRec", {"RMSE": np.nan, "MAE": np.nan, "LLO_HR@10": hr, "LLO_NDCG@10": nd})
        record("SASRec", ranking_metrics(lambda ub: sasrec_scores(sasrec, [train_seq[u] for u in ub])))
        record("Popularity", {"LLO_HR@10": hr_p, "LLO_NDCG@10": nd_p})

## 11. Evaluation: all models on the same test set
RMSE/MAE apply only to rating predictors. Precision@10, Recall@10 and NDCG@10 use relevant = test rating ≥ 4 with train items excluded.
Coverage is the share of the catalogue that appears in at least one user's top-10.

In [ ]:
results = pd.DataFrame(RESULTS).T
cols = [c for c in ["RMSE", "MAE", "Precision@10", "Recall@10", "NDCG@10", "Coverage", "LLO_HR@10", "LLO_NDCG@10"] if c in results]
results = results[cols].astype(float).sort_values("NDCG@10", ascending=False)
results.index.name = "model"
results.round(4).to_csv(f"{OUT_DIR}/results.csv")
print("===== RESULTS (sorted by NDCG@10) =====")
print(results.round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [3, 1]})
results[["Precision@10", "Recall@10", "NDCG@10", "Coverage"]].plot.bar(ax=ax[0], rot=0, width=0.8)
ax[0].set_title("Ranking metrics @10 (higher is better)"); ax[0].set_xlabel("")
rm = results["RMSE"].dropna().sort_values()
ax[1].bar(rm.index, rm.values, color="#C44E52"); ax[1].set_ylim(rm.min() * 0.95, rm.max() * 1.02)
ax[1].set_title("RMSE (lower is better)"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=80); plt.show()

## 12. Demo
A sample user's favourite movies next to each model's top 10, hybrid recommendations with "Because you liked X" explanations,
`similar_movies` for three well-known films, and the edge cases (unknown title, unknown user, user with very few ratings).

In [ ]:
with timed("12. Demo"):
    n_train = pd.Series({u: len(v) for u, v in train_items.items()})
    demo_u = int((n_train - n_train.median()).abs().idxmin())
    demo_user = int(user_ids[demo_u])
    ud = train_df[train_df.u == demo_u].sort_values(["rating", "timestamp"], ascending=False)
    print(f"Demo user {demo_user} ({len(ud)} training ratings). Top-rated movies:")
    display(pd.DataFrame({"title": titles[ud.i.values[:10]], "rating": ud.rating.values[:10]}))

    scorers = {"Popularity": lambda ub: np.tile(wr, (len(ub), 1)), "Content": content_scores, "SVD": svd_scores,
               "Item-KNN": knn_scores, "Hybrid": hybrid_scores, "NeuMF": neumf_scores,
               "Two-tower": lambda ub: tt_scores_for(two_tower, ub)}
    if RUN_SASREC:
        scorers["SASRec"] = lambda ub: sasrec_scores(sasrec, [train_seq[u] for u in ub])
    side = {name: [t[:38] for t in titles[topk_from_scores(fn(np.array([demo_u]))[0], train_items[demo_u], 10)]]
            for name, fn in scorers.items()}
    print("\nTop 10 per model:")
    display(pd.DataFrame({k: pd.Series(v) for k, v in side.items()}))

    print("\nHybrid with explanations:")
    display(explain_hybrid(demo_user))

    for t in ["The Matrix", "godfather", "pulp fiction"]:
        display(similar_movies(t, n=10))

    print("\nEdge cases:")
    print(similar_movies("asdkjhqwe zzz").shape)               # unknown title
    display(recommend_for_user(10 ** 9, n=5))                 # unknown user
    few = n_train.idxmin()
    display(recommend_for_user(int(user_ids[few]), n=5))      # user with the fewest training ratings

## 13. Conclusions
The cell below prints the winners from this run's numbers. The narrative after it explains the patterns.

In [ ]:
best_rank = results["NDCG@10"].idxmax()
print(f"Best ranking model (NDCG@10): {best_rank}  ({results.loc[best_rank, 'NDCG@10']:.4f})")
if results["RMSE"].notna().any():
    best_rmse = results["RMSE"].idxmin()
    print(f"Best rating predictor (RMSE): {best_rmse}  ({results.loc[best_rmse, 'RMSE']:.4f})")
print(f"Highest catalogue coverage: {results['Coverage'].idxmax()} ({results['Coverage'].max():.3f})")

total = time.time() - NOTEBOOK_START
print("\nSection timings:")
for k, v in SECTION_TIMES.items():
    print(f"  {k:32s} {v:7.1f}s")
print(f"TOTAL RUNTIME: {total / 60:.1f} min")
print("Output files:", sorted(f for f in os.listdir(OUT_DIR) if f.endswith((".csv", ".png"))))

**What to take away**

* **Rating prediction vs. ranking are different games.** SVD/KNN/NeuMF minimise RMSE, but low RMSE does not guarantee good top-10 lists:
  ranking the full 9.7k-movie catalogue, rating models push obscure movies with a few high ratings to the top, while the test positives are
  concentrated on popular movies. That is why their Precision/NDCG@10 sits far below their RMSE quality.
* **Popularity is a very strong baseline** on this small, popularity-skewed dataset under a random split, often the one to beat on NDCG@10.
  But it has near-zero coverage and no personalisation; every user gets the same list.
* **Classic vs. deep on small data:** with only about 100k ratings and 610 users, well-regularised classic models (SVD, KNNBaseline) are
  hard to beat on RMSE. Deep models have more parameters than the data supports and need dropout, weight decay and early stopping.
  Their advantages (side features, sequence modelling, GPU-scale retrieval) pay off on larger datasets.
* **Cold start:** content-based scoring and the two-tower item tower (genres + tags) can score movies nobody has rated. Pure CF
  (SVD/KNN/NeuMF) cannot, and new users fall back to popularity.
* **Diversity:** content-based and hybrid models reach further into the long tail (higher coverage). Popularity and ranking-optimised
  models concentrate on the head.
* **Sequential signal:** SASRec needs many interactions per user and many users; with 610 users and a random (not temporal) test split
  its advantage is limited.

**Next steps:** scale to MovieLens 25M (sparse matrices, `implicit`/ALS, approximate nearest-neighbour search such as FAISS for the
two-tower model), use a temporal split, add history features and hard negatives to the two-tower model, blend a popularity prior into the
rating models' rankings, tune the hybrid α on a validation split, and try EASE, LightGCN or BERT4Rec.